In [ ]:
from datasets import load_dataset, DatasetDict

ds = load_dataset("garythung/trashnet")

split1 = ds["train"].train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

ds = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"]
})

print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

dataset-original.zip:   0%|          | 0.00/3.63G [00:00<?, ?B/s]

dataset-resized.zip:   0%|          | 0.00/42.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5054 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 4043
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 505
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 506
    })
})


In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (96, 96)

def preprocess(example):
    image = np.array(example["image"])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    example["image"] = image.numpy()
    return example

ds = ds.map(preprocess)

Map:   0%|          | 0/4043 [00:00<?, ? examples/s]

Map:   0%|          | 0/505 [00:00<?, ? examples/s]

Map:   0%|          | 0/506 [00:00<?, ? examples/s]

In [ ]:
print(ds["train"][0])

{'image': [[[0.9647058844566345, 0.9058823585510254, 0.7843137383460999], [0.9627450704574585, 0.9039215445518494, 0.7823529243469238], [0.9535948038101196, 0.8947712779045105, 0.773202657699585], [0.95686274766922, 0.8980392217636108, 0.7764706015586853], [0.954901933670044, 0.8960784077644348, 0.7745097875595093], [0.9529411792755127, 0.8941176533699036, 0.772549033164978], [0.9490196108818054, 0.8901960849761963, 0.7686274647712708], [0.9509803652763367, 0.8921568393707275, 0.770588219165802], [0.9529411792755127, 0.8941176533699036, 0.772549033164978], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.9529411792755127, 0.9019607901573181, 0.7764706015586853], [0.95686274766922, 0.9058823585510254, 0.7803921699523926], [0.95

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Get train/validation/test data from the split dataset
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

# Convert dataset to arrays
def extract_images_labels(dataset):
    images = []
    labels = []
    for item in dataset:
        images.append(np.array(item["image"]))
        labels.append(item["label"])
    return np.array(images), np.array(labels)

x_train, y_train = extract_images_labels(train_ds)
x_val, y_val = extract_images_labels(val_ds)
x_test, y_test = extract_images_labels(test_ds)

print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

(4043, 96, 96, 3) (4043,)
(505, 96, 96, 3) (505,)
(506, 96, 96, 3) (506,)


In [ ]:
#Simple cnn

In [ ]:
num_classes = len(set(y_train))

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,732,550 (6.61 MB)

 Trainable params: 1,732,550 (6.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Train

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 80s 604ms/step - accuracy: 0.3403 - loss: 1.5524 - val_accuracy: 0.4495 - val_loss: 1.4326
Epoch 2/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 69s 504ms/step - accuracy: 0.5117 - loss: 1.2275 - val_accuracy: 0.5743 - val_loss: 1.2284
Epoch 3/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 82s 507ms/step - accuracy: 0.5879 - loss: 1.0709 - val_accuracy: 0.5960 - val_loss: 1.1037
Epoch 4/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 66s 522ms/step - accuracy: 0.6325 - loss: 0.9691 - val_accuracy: 0.6119 - val_loss: 1.1013
Epoch 5/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 81s 508ms/step - accuracy: 0.6814 - loss: 0.8396 - val_accuracy: 0.6238 - val_loss: 1.0822
Epoch 6/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 64s 504ms/step - accuracy: 0.7475 - loss: 0.6791 - val_accuracy: 0.6515 - val_loss: 1.1243
Epoch 7/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 65s 514ms/step - accuracy: 0.8039 - loss: 0.5439 - val_accuracy: 0.6554 - val_loss: 1.2057
Epoch 8/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 65s 507ms/step - accuracy: 0.8521 - loss: 0

In [ ]:
#Test

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step - accuracy: 0.7213 - loss: 1.2142
Test accuracy: 0.7213438749313354


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 162ms/step - accuracy: 0.7213 - loss: 1.2142
Test accuracy: 0.7213438749313354
Test loss: 1.2141542434692383


In [ ]:
class_names = ds["train"].features["label"].names
num_classes = len(class_names)
print(class_names)
print(num_classes)

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
6


In [ ]:
import numpy as np

def extract_images_labels(dataset):
    images, labels = [], []
    for item in dataset:
        images.append(np.array(item["image"]))
        labels.append(item["label"])
    return np.array(images), np.array(labels)

x_train, y_train = extract_images_labels(ds["train"])
x_val, y_val = extract_images_labels(ds["validation"])
x_test, y_test = extract_images_labels(ds["test"])

print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

(4043, 96, 96, 3) (4043,)
(505, 96, 96, 3) (505,)
(506, 96, 96, 3) (506,)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 96, 96, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=32
)

Epoch 1/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 42s 286ms/step - accuracy: 0.3955 - loss: 1.6196 - val_accuracy: 0.5782 - val_loss: 1.1015
Epoch 2/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 39s 270ms/step - accuracy: 0.5714 - loss: 1.1408 - val_accuracy: 0.6653 - val_loss: 0.9120
Epoch 3/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 35s 272ms/step - accuracy: 0.6394 - loss: 0.9735 - val_accuracy: 0.6950 - val_loss: 0.8324
Epoch 4/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 41s 275ms/step - accuracy: 0.6718 - loss: 0.8829 - val_accuracy: 0.7129 - val_loss: 0.7749
Epoch 5/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 35s 275ms/step - accuracy: 0.6891 - loss: 0.8424 - val_accuracy: 0.7287 - val_loss: 0.7661
Epoch 6/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 43s 289ms/step - accuracy: 0.7183 - loss: 0.7674 - val_accuracy: 0.7366 - val_loss: 0.7345
Epoch 7/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 39s 272ms/step - accuracy: 0.7358 - loss: 0.7403 - val_accuracy: 0.7426 - val_loss: 0.7316
Epoch 8/15
127/127 ━━━━━━━━━━━━━━━━━━━━ 34s 269ms/step - accuracy: 0.7494 - loss: 0

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 170ms/step - accuracy: 0.7826 - loss: 0.6183
Test accuracy: 0.782608687877655
Test loss: 0.6182557940483093


In [ ]:
model.save("ecosort_ai_model.keras")

In [ ]:
from google.colab import files
files.download("ecosort_ai_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf

# Load your saved model
model = tf.keras.models.load_model("ecosort_ai_model.keras")

# Replace with your exact class order
class_names = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

def predict_waste(img):
    if img is None:
        return "No image uploaded"

    img = tf.image.resize(img, (96, 96))
    img = tf.cast(img, tf.float32) / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img, verbose=0)
    class_id = np.argmax(preds[0])
    confidence = float(np.max(preds[0]))

    return f"Prediction: {class_names[class_id]} | Confidence: {confidence:.2f}"

demo = gr.Interface(
    fn=predict_waste,
    inputs=gr.Image(type="numpy", label="Upload waste image"),
    outputs=gr.Textbox(label="Output"),
    title="EcoSort AI",
    description="Upload a waste image and get the predicted class."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f752c9b1be0dc79e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
